# Domain-Adversarial Training (DANN) — CoAtNet-0

Applied to **CoAtNet-0**, the best-performing IPP backbone, to reduce
acquisition-specific bias by making features domain-invariant across
technical replicates.

**Architecture:**
- Feature extractor: CoAtNet-0 backbone
- Biological prediction heads: one linear head per biological attribute
- Domain discriminator: 2-layer MLP attached via a Gradient Reversal Layer (GRL)

**Objective:**
```
L_total = L_bio + L_domain
```
The GRL multiplies gradients by −λ(p) during backprop, where
λ(p) = 2 / (1 + exp(−10p)) − 1 and p ∈ [0,1] is training progress.

The domain variable is the **technical replicate ID** (T1–T24), representing acquisition sessions.

In [ ]:
# !pip install timm --quiet

In [ ]:
import os, random
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import tifffile
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Config
class CFG:
    csv_path  = "../data/slimia_metadata.csv"
    ckpt_dir  = "../checkpoints/ipp/"
    output_dir= "../results/ipp/dann/"

    # Biological attributes only (no microscope/magnification/replicate)
    bio_labels = [
        "cell_line", "culture_medium", "formation_method",
        "seeding_density", "timepoint", "biological_rep", "magnification"
    ]

    train_reps = ["T1", "T2", "T3", "T4"]
    val_reps   = ["T5", "T8"]
    test_reps  = ["T6", "T7"] + [f"T{i}" for i in range(9, 25)]

    image_size  = 224
    batch_size  = 32
    num_epochs  = 200
    patience    = 40
    lr          = 1e-4
    num_workers = 2
    device      = "cuda" if torch.cuda.is_available() else "cpu"
    seeds       = [42, 123, 999]

os.makedirs(CFG.ckpt_dir,   exist_ok=True)
os.makedirs(CFG.output_dir, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

In [ ]:
# Load data
df = pd.read_csv(CFG.csv_path)
df["full_path"] = df["full_path"].astype(str).str.strip()

# Encode biological labels (fit on full dataset)
label_encoders = {}
for col in CFG.bio_labels:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le

# Domain label = technical replicate (acquisition session)
rep_le = LabelEncoder()
df["domain_enc"] = rep_le.fit_transform(df["technical_rep"].astype(str))
n_domains = df["domain_enc"].nunique()
print(f"Number of domain classes (technical reps): {n_domains}")

train_df = df[df["technical_rep"].isin(CFG.train_reps)].copy()
val_df   = df[df["technical_rep"].isin(CFG.val_reps)].copy()
test_df  = df[df["technical_rep"].isin(CFG.test_reps)].copy()

label_dims = {col: df[col + "_enc"].nunique() for col in CFG.bio_labels}
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# Dataset
def load_image_rgb(path):
    try:
        return Image.open(path).convert("RGB")
    except:
        arr = tifffile.imread(path)
        if arr.ndim == 2: arr = np.stack([arr]*3, -1)
        return Image.fromarray(arr.astype(np.uint8)).convert("RGB")


class DANNDataset(Dataset):
    """
    Returns (image, bio_labels [L], domain_label).
    domain_label is the encoded technical replicate ID.
    """
    def __init__(self, frame, transform=None):
        self.df        = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = load_image_rgb(row["full_path"])
        if self.transform: img = self.transform(img)
        bio_labels = torch.tensor([int(row[c + "_enc"]) for c in CFG.bio_labels],
                                  dtype=torch.long)
        domain     = torch.tensor(int(row["domain_enc"]), dtype=torch.long)
        return img, bio_labels, domain


train_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.RandomHorizontalFlip(), T.RandomRotation(10),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
val_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])

train_loader = DataLoader(DANNDataset(train_df, train_tf),
    batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
val_loader   = DataLoader(DANNDataset(val_df, val_tf),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
test_loader  = DataLoader(DANNDataset(test_df, val_tf),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [ ]:
# Gradient Reversal Layer
class GradientReversalFn(torch.autograd.Function):
    """
    Identity in the forward pass.
    Multiplies gradients by −λ in the backward pass.
    λ follows the standard DANN schedule:
        λ(p) = 2 / (1 + exp(−10·p)) − 1
    where p ∈ [0,1] is training progress (epoch / total_epochs).
    """
    @staticmethod
    def forward(ctx, x, lam):
        ctx.save_for_backward(torch.tensor(lam))
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        lam, = ctx.saved_tensors
        return -lam * grad_output, None


class GRL(nn.Module):
    def __init__(self):
        super().__init__()
        self.lam = 0.0

    def forward(self, x):
        return GradientReversalFn.apply(x, self.lam)

    def set_lambda(self, p: float):
        """Update λ according to training progress p ∈ [0,1]."""
        self.lam = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

In [ ]:
# DANN model
class DANNCoAtNet(nn.Module):
    """
    CoAtNet-0 backbone with:
      - biological prediction heads (one per attribute)
      - domain discriminator MLP attached via GRL

    The GRL ensures that the backbone learns domain-invariant features
    by reversing the gradient flow to the discriminator.
    """
    def __init__(self, label_dims: dict, n_domains: int):
        super().__init__()
        self.backbone = timm.create_model("coatnet_0_224", pretrained=False,
                                          num_classes=0)
        D = self.backbone.num_features

        # Biological prediction heads
        self.bio_heads = nn.ModuleDict({
            lab: nn.Linear(D, dim) for lab, dim in label_dims.items()
        })

        # Domain discriminator: Linear → ReLU → Dropout → Linear
        self.grl        = GRL()
        self.discriminator = nn.Sequential(
            nn.Linear(D, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, n_domains)
        )

    def forward(self, x):
        feat     = self.backbone(x)
        bio_outs = {lab: head(feat) for lab, head in self.bio_heads.items()}
        dom_out  = self.discriminator(self.grl(feat))
        return bio_outs, dom_out

In [ ]:
# Training
def train_dann(seed: int):
    set_seed(seed)
    model      = DANNCoAtNet(label_dims, n_domains).to(CFG.device)
    criterions = {c: nn.CrossEntropyLoss() for c in CFG.bio_labels}
    dom_crit   = nn.CrossEntropyLoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5)
    scaler     = torch.amp.GradScaler(enabled=(CFG.device == "cuda"))

    best_f1, patience_count = 0.0, 0
    save_path = os.path.join(CFG.ckpt_dir, f"CoAtNet_DANN_seed{seed}.pth")
    history   = dict(train_bio_f1=[], val_bio_f1=[])

    total_epochs = CFG.num_epochs

    for epoch in range(1, total_epochs + 1):
        # Update λ (GRL schedule)
        p = (epoch - 1) / total_epochs
        model.grl.set_lambda(p)

        # ── Train ──
        model.train()
        t_preds, t_tgts = defaultdict(list), defaultdict(list)

        for imgs, bio_lbls, dom_lbls in tqdm(train_loader,
                                              desc=f"[DANN s{seed}] Ep {epoch:03d}",
                                              leave=False):
            imgs     = imgs.to(CFG.device)
            bio_lbls = bio_lbls.to(CFG.device)
            dom_lbls = dom_lbls.to(CFG.device)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type=CFG.device):
                bio_outs, dom_out = model(imgs)
                bio_loss = sum(criterions[c](bio_outs[c], bio_lbls[:, i])
                               for i, c in enumerate(CFG.bio_labels))
                dom_loss = dom_crit(dom_out, dom_lbls)
                loss     = bio_loss + dom_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            for i, c in enumerate(CFG.bio_labels):
                t_preds[c] += bio_outs[c].argmax(1).detach().cpu().tolist()
                t_tgts[c]  += bio_lbls[:, i].detach().cpu().tolist()

        tr_f1 = np.mean([f1_score(t_tgts[c], t_preds[c],
                                  average="macro", zero_division=0)
                         for c in CFG.bio_labels])

        # Validate 
        model.eval()
        v_preds, v_tgts = defaultdict(list), defaultdict(list)
        with torch.no_grad():
            for imgs, bio_lbls, _ in val_loader:
                imgs     = imgs.to(CFG.device)
                bio_lbls = bio_lbls.to(CFG.device)
                bio_outs, _ = model(imgs)
                for i, c in enumerate(CFG.bio_labels):
                    v_preds[c] += bio_outs[c].argmax(1).cpu().tolist()
                    v_tgts[c]  += bio_lbls[:, i].cpu().tolist()

        va_f1 = np.mean([f1_score(v_tgts[c], v_preds[c],
                                  average="macro", zero_division=0)
                         for c in CFG.bio_labels])
        va_acc = np.mean([accuracy_score(v_tgts[c], v_preds[c])
                          for c in CFG.bio_labels])

        history["train_bio_f1"].append(tr_f1)
        history["val_bio_f1"].append(va_f1)

        scheduler.step(va_f1)
        print(f"  Epoch {epoch:03d} | λ={model.grl.lam:.3f} | "
              f"Train Bio-F1 {tr_f1:.4f} | Val Bio-F1 {va_f1:.4f} "
              f"| Val Acc {va_acc:.4f}")

        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Saved best (F1 {best_f1:.4f})")
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= CFG.patience:
                print(f"  Early stopping.")
                break

    return model, save_path, history


@torch.no_grad()
def eval_dann(model, loader):
    model.eval()
    preds, tgts = defaultdict(list), defaultdict(list)
    for imgs, bio_lbls, _ in loader:
        imgs     = imgs.to(CFG.device)
        bio_lbls = bio_lbls.to(CFG.device)
        bio_outs, _ = model(imgs)
        for i, c in enumerate(CFG.bio_labels):
            preds[c] += bio_outs[c].argmax(1).cpu().tolist()
            tgts[c]  += bio_lbls[:, i].cpu().tolist()
    macro = {}
    for met, fn in [("acc", accuracy_score),
                    ("prec", lambda t,p: precision_score(t,p,average="macro",zero_division=0)),
                    ("rec",  lambda t,p: recall_score(t,p,average="macro",zero_division=0)),
                    ("f1",   lambda t,p: f1_score(t,p,average="macro",zero_division=0))]:
        macro[met] = np.mean([fn(tgts[c], preds[c]) for c in CFG.bio_labels])
    return macro

In [ ]:
# Run 3 seeds
dann_results = []
for seed in CFG.seeds:
    print(f"\n{'='*60}\n  DANN — Seed {seed}\n{'='*60}")
    model, ckpt, history = train_dann(seed)

    # Reload best checkpoint for test evaluation
    model.load_state_dict(torch.load(ckpt, map_location=CFG.device))
    test_m = eval_dann(model, test_loader)
    dann_results.append(test_m)
    print(f"  Test (biological only): {test_m}")

# Summary
print("\n DANN Summary (biological attributes only) ")
for met in ["acc", "prec", "rec", "f1"]:
    vals = [r[met] for r in dann_results]
    print(f"  {met}: {np.mean(vals):.4f} ± {np.std(vals):.5f}")

# Save
pd.DataFrame(dann_results).to_csv(
    os.path.join(CFG.output_dir, "coatnet_dann_results.csv"), index=False)
print("\nResults saved.")

## Interpretability Check: Domain Predictability

After DANN training, we check whether the domain discriminator can still
predict the technical replicate from features — lower accuracy means the
backbone has become more domain-invariant.

In [ ]:
# Evaluate domain discriminator accuracy on val set (last trained model)
model.eval()
dom_preds, dom_tgts = [], []
with torch.no_grad():
    for imgs, _, dom_lbls in val_loader:
        imgs     = imgs.to(CFG.device)
        _, dom_out = model(imgs)
        dom_preds += dom_out.argmax(1).cpu().tolist()
        dom_tgts  += dom_lbls.tolist()

dom_acc = accuracy_score(dom_tgts, dom_preds)
print(f"Domain discriminator accuracy on val set: {dom_acc:.4f}")
print("(Lower = more domain-invariant features)")